# Synthetic Tabular Data Generation - Staged Optimization Driver

**Enhanced notebook for synthetic tabular data generation with staged hyperparameter optimization.**

This notebook provides a staged optimization workflow with:
- Section 1: Setup and imports
- Section 2: Data loading, preprocessing, and EDA
- Section 3: Demo models with default parameters
- Section 4: **Staged Hyperparameter Optimization** (NEW)
  - 4.1: Configuration
  - 4.2: Pilot phase (15 trials) with time estimates
  - 4.3: Continuation options
  - 4.4: Diminishing returns analysis
  - 4.5: Additional batches (optional)
- Section 5: Final models with best parameters

Based on STG-Driver-breast-cancer.ipynb with staged optimization enhancements.

## 1 Setup and Configuration

In [1]:
# Code Chunk ID: CHUNK_1_0_0_A - Import Setup Module
# Import all functionality from setup.py
import sys
sys.path.insert(0, "/home/ec2-user/SageMaker/tableGenCompare")

from setup import *

# Refresh session timestamp to current date
refresh_session_timestamp()

# ============================================================================
# FRESH START CONTROL - Set to True to wipe all checkpoints and start over
# ============================================================================
FRESH_START = True   # <-- Change to True to clear ALL saved progress
# ============================================================================

print("SETUP MODULE IMPORTED SUCCESSFULLY!")
print(f"FRESH_START = {FRESH_START}")
print("=" * 60)

[OK] Essential data science libraries imported successfully!
[OK] Model registry loaded from src/models/registry.py
[OK] Batch training module loaded from src/models/batch_training.py
[OK] Search spaces module loaded from src/models/search_spaces.py
[OK] Batch optimization module loaded from src/models/batch_optimization.py
[OK] Staged optimization module loaded from src/models/staged_optimization.py
[CONFIG] Session timestamp: 2026-03-07
[OK] Parameter management functions loaded from src/utils/parameters.py
[OK] Hyperparameter optimization evaluation functions loaded from src/evaluation/hyperparameters.py
[OK] Optuna objective functions for PATE-GAN and MEDGAN loaded (Phase 5)
Detected sklearn 1.8.0 - applying compatibility patch...
INFO: Applying sklearn compatibility patches for version 1.8.0
Global sklearn compatibility patch applied successfully
CTAB-GAN imported successfully
[OK] CTAB-GAN+ detected and available
[OK] GANerAidModel imported successfully from src.models.implementa

## 2 Data Loading and Pre-processing

### 2.1 Configuration

**USER ATTENTION NEEDED**: Edit the `NOTEBOOK_CONFIG` dictionary below to match your dataset.

In [2]:
# Code Chunk ID: CHUNK_2_1_1_A - NOTEBOOK CONFIGURATION
# ============================================================================
# USER CONFIGURATION - Edit ONLY this block for your dataset
# ============================================================================

NOTEBOOK_CONFIG = {
    # ========== REQUIRED: Dataset Settings ==========
    "data_file": "data/Breast_cancer_data.csv",  # Path to your CSV file
    "target_column": "diagnosis",                 # Target/outcome column name

    # ========== OPTIONAL: Dataset Metadata ==========
    "dataset_name": "Breast Cancer Dataset",      # Display name
    "dataset_identifier_override": None,          # Override auto-detected ID (or None)

    # ========== OPTIONAL: Column Configuration ==========
    # All 5 features are continuous; no categorical columns needed:
    "categorical_columns": [],
    "task_type": "classification",

    # ========== OPTIONAL: Fairness Evaluation ==========
    # Protected attribute for fairness metrics (use cleaned column name after preprocessing).
    # This dataset has no demographic columns, so fairness evaluation is not applicable.
    "protected_col": None,

    # ========== OPTIONAL: Data Subsetting ==========
    "use_row_subset": False,                      # 569 rows - small enough to use all
    "sample_n": 500,                              # Number of rows to sample
    "sample_random_state": 42,                    # Random seed for reproducibility

    # ========== OPTIONAL: Missing Data Handling ==========
    "missing_strategy": "none",                   # No missing values in this dataset
    "mice_max_iter": 10,                          # Max iterations for MICE imputation

    # ========== OPTIONAL: Model Selection ==========
    "models_to_run": ["ctgan", "tvae", "copulagan", "ctabgan", "ctabganplus", "pategan", "medgan"],                       # "all" or list like ["ctgan", "tvae"]
    # "models_to_run": ["ctgan", "tvae", "copulagan", "ctabgan", "ctabganplus", "ganeraid", "pategan", "medgan"],  

    # ========== OPTIONAL: Tuning Configuration ==========
    "tuning_mode": "full",                       # "smoke" (quick) | "full" (thorough)
    "n_trials_smoke": 15,                        # Trials for smoke testing / pilot phase
}

print("NOTEBOOK_CONFIG set successfully!")
print(f"  Data file: {NOTEBOOK_CONFIG['data_file']}")
print(f"  Target column: {NOTEBOOK_CONFIG['target_column']}")
print(f"  Protected column: {NOTEBOOK_CONFIG.get('protected_col', None)}")
print(f"  Models to run: {NOTEBOOK_CONFIG['models_to_run']}")
print(f"  Tuning mode: {NOTEBOOK_CONFIG['tuning_mode']}")

NOTEBOOK_CONFIG set successfully!
  Data file: data/Breast_cancer_data.csv
  Target column: diagnosis
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  Tuning mode: full


### 2.2 Load and Preprocess Data

In [3]:
# Code Chunk ID: CHUNK_2_1_2_A - LOAD AND PREPROCESS DATA
# ============================================================================
# This uses the config-driven preprocessing pipeline
# ============================================================================

if not FRESH_START and 'checkpoint' in dir() and hasattr(checkpoint, 'exists') and checkpoint.exists('section_2'):
    saved = checkpoint.load('section_2')
    data = saved['data']
    original_data = saved['original_data']
    target_column = saved['target_column']
    DATASET_IDENTIFIER = saved['DATASET_IDENTIFIER']
    categorical_columns = saved['categorical_columns']
    metadata = saved['metadata']
    models_to_run = saved['models_to_run']
    RUN_MODE = saved['RUN_MODE']
    TARGET_COLUMN = target_column
    print("[RESUME] Loaded Section 2 from checkpoint")
else:
    # Load and preprocess data using the config
    (
        data,                  # Processed DataFrame
        original_data,         # Copy for reference
        target_column,         # Target column name (cleaned)
        DATASET_IDENTIFIER,    # Dataset identifier for results paths
        categorical_columns,   # List of categorical columns
        metadata               # Full preprocessing metadata
    ) = load_and_preprocess_from_config(NOTEBOOK_CONFIG)

    # Set aliases for backward compatibility with existing notebook cells
    TARGET_COLUMN = target_column

    # Get models to run based on config
    models_to_run = get_models_to_run(NOTEBOOK_CONFIG)

    # Set RUN_MODE for backward compatibility with Section 4 tuning cells
    RUN_MODE = "debug" if NOTEBOOK_CONFIG['tuning_mode'] == "smoke" else "full"

# Initialize checkpoint system (now that DATASET_IDENTIFIER is known)
checkpoint = SectionCheckpoint(DATASET_IDENTIFIER)

# If FRESH_START, wipe all checkpoints and optimization studies
if FRESH_START:
    n_removed = checkpoint.clear_all()
    print(f"[FRESH START] Cleared {n_removed} checkpoint(s) and optimization studies")

existing = checkpoint.list_checkpoints()
if existing:
    print(f"[CHECKPOINT] Found {len(existing)} existing checkpoint(s): {existing}")

# Save Section 2 checkpoint (idempotent - overwrites if exists)
if not checkpoint.exists('section_2'):
    checkpoint.save('section_2', {
        'data': data, 'original_data': original_data,
        'target_column': target_column, 'DATASET_IDENTIFIER': DATASET_IDENTIFIER,
        'categorical_columns': categorical_columns, 'metadata': metadata,
        'models_to_run': models_to_run, 'RUN_MODE': RUN_MODE,
    })

print()
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"  Dataset identifier: {DATASET_IDENTIFIER}")
print(f"  Processed shape: {data.shape}")
print(f"  Target column: {target_column}")
print(f"  Task type: {metadata['task_type']}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Models to run: {models_to_run}")
print(f"  RUN_MODE: {RUN_MODE}")
print(f"  Session: {SESSION_TIMESTAMP}")
print(f"  Results path: {get_results_path(DATASET_IDENTIFIER, 2)}")
print("=" * 60)

[LOAD] Loading data from: data/Breast_cancer_data.csv
[LOAD] Loaded 569 rows, 6 columns
[PREPROCESS] Starting preprocessing pipeline
[PREPROCESS] Input shape: (569, 6)
[PREPROCESS] Categorical columns: []
[PREPROCESS] Task type: classification
[PREPROCESS] Final shape: (569, 6)
[PREPROCESS] Dataset identifier: breast-cancer-data
[FLUSH] Removed 14 pickle file(s) from results/breast-cancer-data/optimization_studies
[FRESH START] Cleared 9 checkpoint(s) and optimization studies

PREPROCESSING COMPLETE
  Dataset identifier: breast-cancer-data
  Processed shape: (569, 6)
  Target column: diagnosis
  Task type: classification
  Categorical columns: []
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  RUN_MODE: full
  Session: 2026-03-07
  Results path: results/breast-cancer-data/2026-03-07/Section-2


### 2.3 Exploratory Data Analysis (EDA)

Comprehensive EDA with automated file export to results directory.

In [4]:
# Code Chunk ID: CHUNK_2_1_EDA - COMPREHENSIVE EDA
# ============================================================================
# Run comprehensive EDA with single function call
# ============================================================================

from src.data.eda import run_comprehensive_eda

eda_results = run_comprehensive_eda(
    data=data,
    target_column=target_column,
    dataset_identifier=DATASET_IDENTIFIER,
    dataset_name=NOTEBOOK_CONFIG.get('dataset_name'),
    categorical_columns=categorical_columns,
    verbose=True
)

# Update categorical_columns from EDA results (in case auto-detected)
categorical_columns = eda_results['categorical_columns']

print(f"\nEDA Results Summary:")
print(f"  Files generated: {len(eda_results['files_generated'])}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Balance ratio: {eda_results.get('balance_ratio', 'N/A')}")

COMPREHENSIVE EXPLORATORY DATA ANALYSIS
Dataset: Breast Cancer Dataset
Target column: diagnosis
Results path: results/breast-cancer-data/2026-03-07/Section-2

[1/6] Dataset Overview...
   Dataset Name.................. Breast Cancer Dataset
   Shape......................... 569 rows x 6 columns
   Memory Usage.................. 0.03 MB
   Total Missing Values.......... 0
   Missing Percentage............ 0.00%
   Duplicate Rows................ 0
   Numeric Columns............... 6
   Categorical Columns........... 0

[2/6] Column Analysis...
   Saved: column_analysis.csv (6 columns)

[3/6] Target Variable Analysis...
   Saved: target_analysis.csv
   Saved: target_balance_metrics.csv
   Balance Ratio: 0.594 (Moderately Imbalanced)

[4/6] Feature Distributions...
   Saved: 1 distribution file(s) (5 features)

[5/6] Correlation Analysis...
   Saved: correlation_heatmap.png
   Saved: correlation_matrix.csv
   Saved: target_correlations.csv (5 features)

[6/6] Configuration Validation...
  

## 3 Demo All Models with Default Parameters

### 3.1 Batch Model Training

In [5]:
# Code Chunk ID: CHUNK_3_1_BATCH
# ============================================================================
# SECTION 3.1 - BATCH MODEL TRAINING
# Train all models configured in NOTEBOOK_CONFIG['models_to_run']
# ============================================================================

from src.models.batch_training import train_models_batch, extract_synthetic_data_to_globals

# Run batch training for all configured models (checkpoint resumes completed models)
training_results = train_models_batch(
    data=data,
    models_to_run=models_to_run,
    target_column=target_column,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    verbose=True,
    checkpoint=checkpoint
)

# Extract synthetic_data_* variables for Section 3.2 compatibility
created_vars = extract_synthetic_data_to_globals(training_results, globals())
print(f"\nCreated variables: {created_vars}")

# Also create model_* variables for backward compatibility
for model_name, result in training_results.items():
    if result['status'] == 'success' and result.get('model') is not None:
        globals()[f'{model_name}_model'] = result['model']


BATCH MODEL TRAINING
Models to train: 7
Dataset shape: (569, 6)
Target column: diagnosis
Samples to generate: 569
GPU available: NVIDIA A10G (22.1 GB)
Device assignments:
  - CTGAN: cuda
  - TVAE: cuda
  - CopulaGAN: cuda
  - CTABGAN: cuda
  - CTABGAN+: cuda
  - PATE-GAN: cuda
  - MEDGAN: cuda


[1/7] Training CTGAN...
--------------------------------------------------
  Device: cuda
  Training CTGAN...


Gen. (-0.98) | Discrim. (0.06): 100%|██████████| 300/300 [00:04<00:00, 65.97it/s] 


  Generating 569 synthetic samples...
  [OK] CTGAN completed in 10.46s
  Synthetic data shape: (569, 6)

[2/7] Training TVAE...
--------------------------------------------------
  Device: cuda
  Training TVAE...
  Generating 569 synthetic samples...
[OK] Target integrity functions loaded from src/data/target_integrity.py
  [OK] TVAE completed in 5.04s
  Synthetic data shape: (569, 6)

[3/7] Training CopulaGAN...
--------------------------------------------------
  Device: cuda
  Training CopulaGAN...
  Generating 569 synthetic samples...
  [OK] CopulaGAN completed in 6.19s
  Synthetic data shape: (569, 6)

[4/7] Training CTABGAN...
--------------------------------------------------
  Device: cuda
  Training CTABGAN...


100%|██████████| 300/300 [00:14<00:00, 21.40it/s]


Finished training in 14.765811681747437  seconds.
  Generating 569 synthetic samples...
  [OK] CTABGAN completed in 14.80s
  Synthetic data shape: (569, 6)

[5/7] Training CTABGAN+...
--------------------------------------------------
  Device: cuda
  Training CTABGAN+...


100%|██████████| 400/400 [00:13<00:00, 30.02it/s]


Finished training in 14.069697380065918  seconds.
  Generating 569 synthetic samples...
  [OK] CTABGAN+ completed in 14.10s
  Synthetic data shape: (569, 6)

[6/7] Training PATE-GAN...
--------------------------------------------------
  Device: cuda
  Training PATE-GAN...
  Generating 569 synthetic samples...
  [OK] PATE-GAN completed in 2.08s
  Synthetic data shape: (569, 6)

[7/7] Training MEDGAN...
--------------------------------------------------
  Device: cuda
  Training MEDGAN...
  Generating 569 synthetic samples...
  [OK] MEDGAN completed in 5.83s
  Synthetic data shape: (569, 6)

BATCH TRAINING SUMMARY
Total models: 7
Successful: 7
Failed: 0

Successful models:
  - CTGAN: 10.46s
  - TVAE: 5.04s
  - CopulaGAN: 6.19s
  - CTABGAN: 14.80s
  - CTABGAN+: 14.10s
  - PATE-GAN: 2.08s
  - MEDGAN: 5.83s

Created variables: ['synthetic_data_ctgan', 'synthetic_data_tvae', 'synthetic_data_copulagan', 'synthetic_data_ctabgan', 'synthetic_data_ctabganplus', 'synthetic_data_pategan', 'synthe

### 3.2 Batch Evaluation

In [6]:
# Code Chunk ID: CHUNK_3_2_0_A
# ============================================================================
# SECTION 3.2 - BATCH EVALUATION FOR ALL TRAINED MODELS
# ============================================================================

print("SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION")
print("=" * 60)

if checkpoint.exists('section_3.2'):
    section3_results = checkpoint.load('section_3.2')['results']
    print("[RESUME] Loaded Section 3.2 evaluation from checkpoint")
else:
    section3_results = evaluate_all_available_models(
        section_number=3,
        scope=globals(),
        models_to_evaluate=None,
        real_data=None,
        target_col=None,
        protected_col=NOTEBOOK_CONFIG.get("protected_col")
    )
    checkpoint.save('section_3.2', {'results': section3_results})

if section3_results:
    print(f"\nSECTION 3 BATCH EVALUATION COMPLETED!")
    print(f"Evaluated {len(section3_results)} models successfully")
    
    # Show ranking by quality score
    best_models = []
    for model_name, results in section3_results.items():
        if 'error' not in results:
            quality_score = results.get('overall_quality_score', 0)
            best_models.append((model_name, quality_score))
    
    if best_models:
        best_models.sort(key=lambda x: x[1], reverse=True)
        print(f"\nRANKING BY QUALITY SCORE:")
        for i, (model, score) in enumerate(best_models, 1):
            print(f"   {i}. {model}: {score:.3f}")
else:
    print("\nNo models available for evaluation")

SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION
[OK] Batch evaluation functions loaded from src/evaluation/batch.py
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 3
[INFO] Dataset: breast-cancer-data
[INFO] Target column: diagnosis
[INFO] Protected column: None (fairness metrics skipped)
[INFO] MIA evaluation: OFF
[INFO] Variable pattern: standard
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/breast-cancer-data/2026-03-07/Section-3/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.602

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity

## 4 Staged Hyperparameter Optimization

This section uses a staged approach to hyperparameter optimization:

- **Smoke mode** (`tuning_mode="smoke"`): Runs 10 pilot trials per model, then displays
  time-budget recommendations (how many trials fit in 1 hour, how long 20 trials would take).
  Section 5 uses the pilot results directly.
- **Full mode** (`tuning_mode="full"`): Runs a pilot phase, displays time estimates, then
  presents three continuation strategies in cell 4.3.  The user reviews the estimates and
  **uncomments one option** before running the cell.

### 4.1 Configuration

Create the `StagedOptimizationManager`.  `TUNING_MODE` and `PILOT_TRIALS` are derived
from `NOTEBOOK_CONFIG` so the same code works for both smoke and full runs.

In [7]:
# Code Chunk ID: CHUNK_4_1_CONFIG
# ============================================================================
# SECTION 4.1 - STAGED OPTIMIZATION CONFIGURATION
# ============================================================================

from src.models.staged_optimization import (
    StagedOptimizationManager,
    StagedOptimizationConfig,
    flush_previous_runs
)

# Flush optimization studies if FRESH_START is set
if FRESH_START:
    flush_previous_runs(DATASET_IDENTIFIER)

# Derive tuning mode and pilot size from NOTEBOOK_CONFIG
TUNING_MODE = NOTEBOOK_CONFIG.get("tuning_mode", "smoke")
PILOT_TRIALS = 10 if TUNING_MODE == "smoke" else NOTEBOOK_CONFIG.get("n_trials_smoke", 10)

# Configure staged optimization
staged_config = StagedOptimizationConfig(
    pilot_trials=PILOT_TRIALS,
    verbose_level=1,           # 0=silent, 1=concise, 2=detailed
    persistence_dir=f"results/{DATASET_IDENTIFIER}/optimization_studies",
    run_mode=RUN_MODE,         # "debug" or "full"
    random_state=42,
    continue_on_error=True
)

# Create the manager
manager = StagedOptimizationManager(
    data=data,
    target_column=target_column,
    categorical_columns=categorical_columns,
    dataset_identifier=DATASET_IDENTIFIER,
    config=staged_config
)

print("Staged Optimization Manager created!")
print(f"  Tuning mode: {TUNING_MODE}")
print(f"  Pilot trials: {staged_config.pilot_trials}")
print(f"  Run mode: {staged_config.run_mode}")
print(f"  Persistence dir: {staged_config.persistence_dir}")
print(f"  FRESH_START: {FRESH_START}")

[FLUSH] No previous studies found in results/breast-cancer-data/optimization_studies — starting clean
Staged Optimization Manager created!
  Tuning mode: full
  Pilot trials: 15
  Run mode: full
  Persistence dir: results/breast-cancer-data/optimization_studies
  FRESH_START: True


### 4.2 Run Pilot Phase

Run a pilot phase to establish time estimates.

- **Smoke mode**: 10 trials + smoke recommendations table (trials in 1 hr, time for 20 trials).
- **Full mode**: Pilot trials + time estimates for planning continuation.

In [8]:
# Code Chunk ID: CHUNK_4_2_PILOT
# ============================================================================
# SECTION 4.2 - RUN PILOT PHASE
# ============================================================================

# Run pilot phase (uses PILOT_TRIALS from cell 4.1)
pilot_states = manager.run_pilot(
    models_to_run=models_to_run
)

# Display time estimates
print("\n" + "="*60)
print("PILOT PHASE COMPLETE - TIME ESTIMATES")
print("="*60)

time_estimates = manager.get_time_estimates()
if len(time_estimates) > 0:
    print(time_estimates.to_string(index=False))
else:
    print("No time estimates available")

# Show best scores so far
print("\nBest scores after pilot:")
for model_name, state in pilot_states.items():
    print(f"  {model_name}: {state.best_score:.4f} ({state.total_trials} trials)")

# Smoke mode: show budget recommendations
if TUNING_MODE == "smoke":
    print("\n" + "="*60)
    print("SMOKE MODE - TIME BUDGET RECOMMENDATIONS")
    print("="*60)
    smoke_recs = manager.get_smoke_recommendations()
    print(smoke_recs.to_string(index=False))
    print("\nTo run a full optimization, set tuning_mode='full' in NOTEBOOK_CONFIG")
    print("and use the recommended_pilot column to guide n_trials_smoke.")

[I 2026-03-07 15:42:19,836] A new study created in memory with name: ctgan_hpo_breast-cancer-data



STAGED OPTIMIZATION - PILOT PHASE
Models to optimize: 7
Pilot trials per model: 15
Dataset identifier: breast-cancer-data


[PILOT] Optimizing CTGAN...
--------------------------------------------------


Gen. (-0.68) | Discrim. (0.12): 100%|██████████| 200/200 [00:05<00:00, 33.83it/s] 


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.3745
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5044
[CHART] Combined Score: 0.4265 (Similarity: 0.3745, Accuracy: 0.5044)
[ctgan] Trial 1: Combined Score: 0.4265 (Similarity: 0.3745, Accuracy: 0.5044) Best Score so far: 0.4265


Gen. (-0.67) | Discrim. (0.09): 100%|██████████| 200/200 [00:05<00:00, 35.66it/s] 


...

Finished training in 42.703736782073975  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5837
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8893
[CHART] Combined Score: 0.7059 (Similarity: 0.5837, Accuracy: 0.8893)
[ctabganplus] Trial 13: Combined Score: 0.7059 (Similarity: 0.5837, Accuracy: 0.8893) Best Score so far: 0.7391


100%|██████████| 900/900 [00:42<00:00, 21.42it/s]


Finished training in 42.76183748245239  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5437
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8972
[CHART] Combined Score: 0.6851 (Similarity: 0.5437, Accuracy: 0.8972)
[ctabganplus] Trial 14: Combined Score: 0.6851 (Similarity: 0.5437, Accuracy: 0.8972) Best Score so far: 0.7391


100%|██████████| 450/450 [00:21<00:00, 21.04it/s]


Finished training in 22.13834309577942  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5903
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8981
[CHART] Combined Score: 0.7134 (Similarity: 0.5903, Accuracy: 0.8981)
[ctabganplus] Trial 15: Combined Score: 0.7134 (Similarity: 0.5903, Accuracy: 0.8981) Best Score so far: 0.7391
  [OK] CTABGAN+: 15 trials in 870.3s
  Best score: 0.7391

[PILOT] Optimizing PATE-GAN...
--------------------------------------------------
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.2542
[OK] TRTS Evaluation: 2 scenarios, Average: 0.1863
[CHART] Combined Score: 0.2271 (Similarity: 0.2542, Accuracy: 0.1863)
[pategan] Trial 1: Combined Score: 0.2271 (Similarity: 0.2542, Accuracy: 0.1863) Best Score so far: 0.2271
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity A

In [9]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS
# ============================================================================
print("DIMINISHING RETURNS ANALYSIS")
print("="*60)

convergence_report = manager.get_diminishing_returns_report()

if len(convergence_report) > 0:
    print(convergence_report.to_string(index=False))
    
    print("\nInterpretation:")
    print("-" * 40)
    for _, row in convergence_report.iterrows():
        print(f"  {row['model']}: {row['recommendation']}")
        if row['has_plateaued']:
            print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
        else:
            print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
else:
    print("No convergence data available")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued                         recommendation
      ctgan      15    0.644213          0.002244           0.217739           True                 Stop - plateau reached
       tvae      15    0.711609          0.000000           0.076236           True                 Stop - plateau reached
  copulagan      15    0.671208          0.048405           0.108306          False Consider stopping - minor improvements
    ctabgan      15    0.727558          0.000000           0.096303           True                 Stop - plateau reached
ctabganplus      15    0.739107          0.000000           0.070230           True                 Stop - plateau reached
    pategan      15    0.468924          0.000000           0.241874           True                 Stop - plateau reached
     medgan      15    0.448812          0.000000           0.122694           True                 Stop - pla

### 4.3 Continuation (Full Mode Only)

**Full mode only** - Review the pilot results and time estimates above, then
uncomment **one** of the three options below and adjust the values before running.

- **Option (i)**: Common trial count for all models
- **Option (ii)**: Per-model trial counts
- **Option (iii)**: Time-based budget (minutes per model)

Models not in `models_to_run` are silently ignored, so listing all 8 is safe.

In [10]:
# Code Chunk ID: CHUNK_4_3_CONTINUE
# ============================================================================
# SECTION 4.3 - CONTINUATION (Full mode only - choose ONE option)
# ============================================================================

if TUNING_MODE == "smoke":
    print("[SMOKE MODE] Skipping continuation - using pilot results for Section 5.")
else:
    # ---- Uncomment ONE option below, adjust values, then run this cell ----

    # OPTION (i): Common trials for all models
    # continuation_states = manager.continue_optimization(additional_trials=30)

    # OPTION (ii): Per-model trials - adjust counts per model
    continuation_states = manager.continue_optimization(
        trials_per_model={
            'ctgan': 10,
            'tvae':10,
            'copulagan': 10,
            'ctabgan': 10,
            'ctabganplus': 10,
            'ganeraid': 10,
            'pategan': 10,
            'medgan': 10,
        }
    )

    # OPTION (iii): Time-based budget - minutes per model
    # continuation_states = manager.continue_optimization(
    #     time_budget_minutes={
    #         'ctgan': 60,
    #         'tvae': 60,
    #         'copulagan': 60,
    #         'ctabgan': 60,
    #         'ctabganplus': 60,
    #         'ganeraid': 60,
    #         'pategan': 60,
    #         'medgan': 60,
    #     }
    # )

    print("[FULL MODE] Uncomment one option above and re-run this cell.")


STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 10 additional trials
  tvae: 10 additional trials
  copulagan: 10 additional trials
  ctabgan: 10 additional trials
  ctabganplus: 10 additional trials
  ganeraid: 10 additional trials
  pategan: 10 additional trials
  medgan: 10 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------
  Resuming from 15 existing trials


Gen. (-0.14) | Discrim. (-0.24): 100%|██████████| 900/900 [00:25<00:00, 34.68it/s]


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4866
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8111
[CHART] Combined Score: 0.6164 (Similarity: 0.4866, Accuracy: 0.8111)
[ctgan] Trial 16: Combined Score: 0.6164 (Similarity: 0.4866, Accuracy: 0.8111) Best Score so far: 0.6442


Gen. (-0.86) | Discrim. (-0.00): 100%|██████████| 350/350 [00:11<00:00, 31.80it/s]


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.3949
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5404
[CHART] Combined Score: 0.4531 (Similarity: 0.3949, Accuracy: 0.5404)
[ctgan] Trial 17: Combined Score: 0.4531 (Similarity: 0.3949, Accuracy: 0.5404) Best Score so far: 0.6442


...

Finished training in 45.47180247306824  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5764
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9051
[CHART] Combined Score: 0.7079 (Similarity: 0.5764, Accuracy: 0.9051)
[ctabganplus] Trial 23: Combined Score: 0.7079 (Similarity: 0.5764, Accuracy: 0.9051) Best Score so far: 0.7511


100%|██████████| 850/850 [00:40<00:00, 21.23it/s]


Finished training in 40.8020281791687  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5700
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8893
[CHART] Combined Score: 0.6977 (Similarity: 0.5700, Accuracy: 0.8893)
[ctabganplus] Trial 24: Combined Score: 0.6977 (Similarity: 0.5700, Accuracy: 0.8893) Best Score so far: 0.7511


100%|██████████| 1000/1000 [00:47<00:00, 21.21it/s]


Finished training in 47.90981411933899  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5701
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8963
[CHART] Combined Score: 0.7006 (Similarity: 0.5701, Accuracy: 0.8963)
[ctabganplus] Trial 25: Combined Score: 0.7006 (Similarity: 0.5701, Accuracy: 0.8963) Best Score so far: 0.7511
  [OK] CTABGAN+: 10 trials in 379.3s
  Best score: 0.7511

[CONTINUE] Optimizing PATE-GAN...
--------------------------------------------------
  Resuming from 15 existing trials
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.2749
[PRUNED] Trial pruned after similarity calculation (score: 0.2749)
[pategan] Trial 16: Combined Score: 0.2749 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.4689
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 va

In [11]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS (post-continuation)
# ============================================================================

if TUNING_MODE == "full":
    print("DIMINISHING RETURNS ANALYSIS")
    print("="*60)

    convergence_report = manager.get_diminishing_returns_report()

    if len(convergence_report) > 0:
        print(convergence_report.to_string(index=False))

        print("\nInterpretation:")
        print("-" * 40)
        for _, row in convergence_report.iterrows():
            print(f"  {row['model']}: {row['recommendation']}")
            if row['has_plateaued']:
                print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
            else:
                print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
    else:
        print("No convergence data available")
else:
    print("[SMOKE MODE] Skipping post-continuation analysis.")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued                         recommendation
      ctgan      25    0.644213          0.000000           0.217739           True                 Stop - plateau reached
       tvae      25    0.715323          0.000000           0.079950           True                 Stop - plateau reached
  copulagan      25    0.671208          0.000000           0.108306           True                 Stop - plateau reached
    ctabgan      25    0.727590          0.000000           0.096335           True                 Stop - plateau reached
ctabganplus      25    0.751112          0.015983           0.082235          False Consider stopping - minor improvements
    pategan      25    0.468924          0.000000           0.241874           True                 Stop - plateau reached
     medgan      25    0.448812          0.000000           0.122694           True                 Stop - pla

### 4.5 Additional Batches (Full Mode, Optional)

If the diminishing returns analysis suggests continuing, uncomment one option below.
You can re-run this cell multiple times.

In [12]:
# Code Chunk ID: CHUNK_4_5_ADDITIONAL
# ============================================================================
# SECTION 4.5 - ADDITIONAL BATCHES (Full mode only - optional, can repeat)
# ============================================================================

if TUNING_MODE == "smoke":
    print("[SMOKE MODE] Skipping additional batches.")
else:
    # ---- Uncomment ONE option below, adjust values, then run this cell ----

    # OPTION (i): Common trials for all models
    # additional_states = manager.continue_optimization(additional_trials=20)

    # OPTION (ii): Per-model trials - adjust counts per model
    additional_states = manager.continue_optimization(
          trials_per_model={
              'ctgan': 1,
              'tvae': 1,
              'copulagan': 1,
              'ctabgan': 1,
              'ctabganplus': 1,
              'ganeraid': 1,
              'pategan': 1,
              'medgan': 1,
          }
      )

    # OPTION (iii): Time-based budget - minutes per model
    # additional_states = manager.continue_optimization(
    #     time_budget_minutes={
    #         'ctgan': 30,
    #         'tvae': 30,
    #         'copulagan': 30,
    #         'ctabgan': 30,
    #         'ctabganplus': 30,
    #         'ganeraid': 30,
    #         'pategan': 30,
    #         'medgan': 30,
    #     }
    # )

    # print("\nUpdated convergence report:")
    # print(manager.get_diminishing_returns_report().to_string(index=False))

    print("Cell skipped - uncomment an option above to run additional batches")


STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 1 additional trials
  tvae: 1 additional trials
  copulagan: 1 additional trials
  ctabgan: 1 additional trials
  ctabganplus: 1 additional trials
  ganeraid: 1 additional trials
  pategan: 1 additional trials
  medgan: 1 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------
  Resuming from 25 existing trials


Gen. (-0.20) | Discrim. (-0.21): 100%|██████████| 800/800 [00:22<00:00, 34.81it/s]


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4720
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7979
[CHART] Combined Score: 0.6024 (Similarity: 0.4720, Accuracy: 0.7979)
[ctgan] Trial 26: Combined Score: 0.6024 (Similarity: 0.4720, Accuracy: 0.7979) Best Score so far: 0.6442
  [OK] CTGAN: 1 trials in 27.7s
  Best score: 0.6442

[CONTINUE] Optimizing TVAE...
--------------------------------------------------
  Resuming from 25 existing trials
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4695
[PRUNED] Trial pruned after similarity calculation (score: 0.4695)
[tvae] Trial 26: Combined Score: 0.4695 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.7153
  [OK] TVAE: 1 trials in 6.5s
  Best score: 0.7153

[CONTINUE] Optimizing CopulaGAN...
--------------------------------------------------
  Resuming from 25 existing tri

100%|██████████| 600/600 [00:36<00:00, 16.28it/s]


Finished training in 37.617876052856445  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5623
[PRUNED] Trial pruned after similarity calculation (score: 0.5623)
[ctabgan] Trial 26: Combined Score: 0.5623 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.7276
  [OK] CTABGAN: 1 trials in 37.7s
  Best score: 0.7276

[CONTINUE] Optimizing CTABGAN+...
--------------------------------------------------
  Resuming from 25 existing trials


100%|██████████| 900/900 [00:56<00:00, 15.99it/s]


Finished training in 57.172114610672  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5738
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8849
[CHART] Combined Score: 0.6983 (Similarity: 0.5738, Accuracy: 0.8849)
[ctabganplus] Trial 26: Combined Score: 0.6983 (Similarity: 0.5738, Accuracy: 0.8849) Best Score so far: 0.7511
  [OK] CTABGAN+: 1 trials in 57.4s
  Best score: 0.7511

[CONTINUE] Optimizing PATE-GAN...
--------------------------------------------------
  Resuming from 25 existing trials
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.2853
[PRUNED] Trial pruned after similarity calculation (score: 0.2853)
[pategan] Trial 26: Combined Score: 0.2853 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.4689
  [OK] PATE-GAN: 1 trials in 2.0s
  Best score: 0.4689

[CONTINUE] Optimizing MEDGAN...
------------------

### 4.6 Save Best Parameters

In [13]:
# Code Chunk ID: CHUNK_4_6_SAVE
# ============================================================================
# SECTION 4.6 - SAVE BEST PARAMETERS TO CSV
# ============================================================================

from src.utils.parameters import save_best_parameters_to_csv

# Extract studies from manager to globals for saving
for model_name, state in manager.model_states.items():
    if state.study is not None:
        globals()[f"{model_name}_study"] = state.study

# Save all best parameters to CSV for Section 5
save_result = save_best_parameters_to_csv(
    scope=globals(),
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER
)

print(f"\nParameters saved: {save_result['success']}")
print(f"Files: {save_result.get('files_saved', [])}")

[SAVE] SAVING BEST PARAMETERS FROM SECTION 4
[FOLDER] Target directory: results/breast-cancer-data/2026-03-07/Section-4

[CHART] Processing CTGAN parameters...
[OK] Found CTGAN: 10 parameters, score: 0.6442

[CHART] Processing CTAB-GAN parameters...
[OK] Found CTAB-GAN: 2 parameters, score: 0.7276

[CHART] Processing CTAB-GAN+ parameters...
[OK] Found CTAB-GAN+: 2 parameters, score: 0.7511

[CHART] Processing GANerAid parameters...
[WARNING]  GANerAid: Study variable 'ganeraid_study' not found

[CHART] Processing CopulaGAN parameters...
[OK] Found CopulaGAN: 6 parameters, score: 0.6712

[CHART] Processing TVAE parameters...
[OK] Found TVAE: 8 parameters, score: 0.7153

[CHART] Processing PATE-GAN parameters...
[OK] Found PATE-GAN: 10 parameters, score: 0.4689

[CHART] Processing MEDGAN parameters...
[OK] Found MEDGAN: 11 parameters, score: 0.4488

[OK] Parameters saved: results/breast-cancer-data/2026-03-07/Section-4/best_parameters.csv
   - Total parameter entries: 67
[OK] Summary sav

## 5 Final Model Comparison with Best Parameters

### 5.1 Train All Models with Best Parameters from Section 4

In [14]:
# Code Chunk ID: CHUNK_5_1_BATCH
# ============================================================================
# SECTION 5.1 - BATCH TRAINING WITH BEST PARAMETERS
# ============================================================================

from src.models.batch_training import train_models_batch_with_best_params

# Train all models with best parameters from Section 4 (checkpoint resumes completed models)
section5_results = train_models_batch_with_best_params(
    data=data,
    target_column=target_column,
    models_to_run=models_to_run,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER,
    scope=globals(),
    verbose=True,
    checkpoint=checkpoint
)

# Show summary of created variables
final_vars = [k for k in globals().keys() if k.endswith('_final')]
print(f"\nFinal synthetic data variables: {sorted(final_vars)}")


SECTION 5.1: BATCH TRAINING WITH BEST PARAMETERS
Models to train: 7
Dataset shape: (569, 6)
Target column: diagnosis
Samples to generate: 569
Loading parameters from: Section 4
GPU available: NVIDIA A10G (22.1 GB)
Device assignments:
  - CTGAN: cuda
  - TVAE: cuda
  - CopulaGAN: cuda
  - CTABGAN: cuda
  - CTABGAN+: cuda
  - PATE-GAN: cuda
  - MEDGAN: cuda

[1/3] Loading best parameters from Section 4...
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/breast-cancer-data/2026-03-07/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 10 parameters
[OK] Loaded CTAB-GAN: 2 parameters
[OK] Loaded CTAB-GAN+: 2 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 8 parameters
[OK] Loaded PATE-GAN: 10 parameters
[OK] Loaded MEDGAN: 11 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 7
   - ctgan: 10 parameters
   - ctabgan: 2 parameters
   - ctabganplus: 2 parameters
   - copulaga

Gen. (-0.49) | Discrim. (0.05): 100%|██████████| 1000/1000 [00:20<00:00, 48.80it/s]


  Generating 569 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5334
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8541
[CHART] Combined Score: 0.6617 (Similarity: 0.5334, Accuracy: 0.8541)
  [OK] CTGAN completed in 21.56s
  Synthetic data shape: (569, 6)
  Objective score: 0.6617

[2/7] Training TVAE...
--------------------------------------------------
  Device: cuda
  Parameters loaded: 8
    - epochs: 700
    - batch_size: 256
    - learning_rate: 0.0022
    - embedding_dim: 192
    - l2scale: 0.0012
    ... and 4 more
  Training TVAE...
  Generating 569 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5363
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8998
[CHART] Combined Score: 0.6817 (Similarity: 0.5363, Accuracy: 0.8998)
  [OK] TVAE completed in 14.55s
  Synthetic data shape: (569, 

100%|██████████| 550/550 [00:29<00:00, 18.63it/s]


Finished training in 30.293522119522095  seconds.
  Generating 569 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.6215
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8998
[CHART] Combined Score: 0.7328 (Similarity: 0.6215, Accuracy: 0.8998)
  [OK] CTABGAN completed in 30.33s
  Synthetic data shape: (569, 6)
  Objective score: 0.7328

[5/7] Training CTABGAN+...
--------------------------------------------------
  Device: cuda
  Parameters loaded: 2
    - epochs: 1000
    - batch_size: 512
    - categorical_columns: []
    - target_col: diagnosis
  Training CTABGAN+...


100%|██████████| 1000/1000 [00:54<00:00, 18.21it/s]


Finished training in 55.66142296791077  seconds.
  Generating 569 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.6334
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8928
[CHART] Combined Score: 0.7372 (Similarity: 0.6334, Accuracy: 0.8928)
  [OK] CTABGAN+ completed in 55.70s
  Synthetic data shape: (569, 6)
  Objective score: 0.7372

[6/7] Training PATE-GAN...
--------------------------------------------------
  Device: cuda
  Parameters loaded: 10
    - epochs: 100
    - batch_size: 32
    - num_teachers: 15
    - generator_lr: 0.0000
    - discriminator_lr: 0.0007
    ... and 6 more
  Training PATE-GAN...
  Generating 569 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.3391
[ERROR] TRTS (Real->Synthetic) failed: Classification metrics can't handle a mix of continuous and binary targets
[ER

### 5.2 Batch Evaluation of Optimized Models

In [15]:
# Code Chunk ID: CHUNK_5_2_0_A
# ============================================================================
# SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
# ============================================================================

print("SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION")
print("=" * 80)

from setup import evaluate_section5_optimized_models

if checkpoint.exists('section_5.2'):
    section5_batch_results = checkpoint.load('section_5.2')['results']
    print("[RESUME] Loaded Section 5.2 evaluation from checkpoint")
    print(f"Models processed: {section5_batch_results['models_processed']}")
    print(f"Results exported to: {section5_batch_results['results_dir']}")
else:
    try:
        section5_batch_results = evaluate_section5_optimized_models(
            section_number=5,
            scope=globals(),
            target_column=TARGET_COLUMN,
            protected_col=NOTEBOOK_CONFIG.get("protected_col"),
            compute_mia=True
        )
        checkpoint.save('section_5.2', {'results': section5_batch_results})
        
        print("\n" + "="*80)
        print("SECTION 5.2 OPTIMIZED MODELS BATCH EVALUATION COMPLETED!")
        print("="*80)
        print(f"Models processed: {section5_batch_results['models_processed']}")
        print(f"Results exported to: {section5_batch_results['results_dir']}")
        
    except Exception as e:
        print(f"Section 5.2 batch evaluation failed: {e}")
        import traceback
        traceback.print_exc()

SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 5
[INFO] Dataset: breast-cancer-data
[INFO] Target column: diagnosis
[INFO] Protected column: None (fairness metrics skipped)
[INFO] MIA evaluation: OFF
[INFO] Variable pattern: final
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/breast-cancer-data/2026-03-07/Section-5/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.891

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.017
   [CHART] Explained Variance (PC1, PC2): 0.638, 0.172

[3] 

### 5.3 Final Summary

In [16]:
# Code Chunk ID: CHUNK_5_3_SUMMARY
# ============================================================================
# SECTION 5.3 - FINAL SUMMARY
# ============================================================================

print("=" * 80)
print("SYNTHETIC DATA GENERATION - FINAL SUMMARY")
print("=" * 80)
print(f"\nDataset: {DATASET_IDENTIFIER}")
print(f"Session: {SESSION_TIMESTAMP}")
print(f"\nResults directories:")
print(f"  Section 2 (EDA): {get_results_path(DATASET_IDENTIFIER, 2)}")
print(f"  Section 3 (Demo): {get_results_path(DATASET_IDENTIFIER, 3)}")
print(f"  Section 4 (HPO): {get_results_path(DATASET_IDENTIFIER, 4)}")
print(f"  Section 5 (Final): {get_results_path(DATASET_IDENTIFIER, 5)}")

# Show staged optimization summary
if 'manager' in dir() and manager is not None:
    print(f"\nStaged Optimization Summary:")
    print("-" * 60)
    summary = manager.get_best_params_summary()
    if len(summary) > 0:
        print(summary[['model', 'best_score', 'total_trials', 'status']].to_string(index=False))

# Show final model performance
if 'section5_results' in dir() and section5_results:
    print(f"\nFinal Model Performance (with optimized parameters):")
    print("-" * 60)
    
    scores = []
    for model_name, result in section5_results.items():
        if result['status'] == 'success':
            score = result.get('objective_score', 0)
            time_taken = result.get('training_time', 0)
            scores.append((model_name, score, time_taken))
    
    # Sort by score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    
    for i, (model, score, time_taken) in enumerate(scores, 1):
        print(f"  {i}. {model.upper()}: score={score:.4f}, time={time_taken:.1f}s")
    
    if scores:
        best_model = scores[0][0]
        print(f"\nBest performing model: {best_model.upper()}")
        print(f"Best synthetic data variable: synthetic_{best_model}_final")

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETE")
print("=" * 80)

SYNTHETIC DATA GENERATION - FINAL SUMMARY

Dataset: breast-cancer-data
Session: 2026-03-07

Results directories:
  Section 2 (EDA): results/breast-cancer-data/2026-03-07/Section-2
  Section 3 (Demo): results/breast-cancer-data/2026-03-07/Section-3
  Section 4 (HPO): results/breast-cancer-data/2026-03-07/Section-4
  Section 5 (Final): results/breast-cancer-data/2026-03-07/Section-5

Staged Optimization Summary:
------------------------------------------------------------
      model  best_score  total_trials    status
      ctgan    0.644213            26 completed
       tvae    0.715323            26 completed
  copulagan    0.671208            26 completed
    ctabgan    0.727590            26 completed
ctabganplus    0.751112            26 completed
    pategan    0.468924            26 completed
     medgan    0.448812            26 completed

Final Model Performance (with optimized parameters):
------------------------------------------------------------
  1. CTABGANPLUS: score=0.